# 03 - Calibration and trustworthy probabilities

Purpose: test whether a predicted probability means what it says. Ranking metrics alone are not enough for a retention budget.

Brier score is the main probability metric: lower is better. The reliability curve compares average predicted risk with observed churn.

In [ ]:
from pathlib import Path
import pickle
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ARTIFACTS = ROOT / 'artifacts'
FIGURES = ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)

# Use the benchmark winner if notebook 02 has been run; otherwise use the baseline model.
if (ARTIFACTS / 'benchmark_models.pkl').exists():
    with open(ARTIFACTS / 'benchmark_models.pkl', 'rb') as handle:
        bundle = pickle.load(handle)
    model_name = (
        bundle['results']
        .sort_values('PR-AUC', ascending=False)
        .iloc[0]['model']
    )
    model = bundle['models'][model_name]
    X_test = bundle['X_test']
    y_test = bundle['y_test']
else:
    bundle = joblib.load(ARTIFACTS / 'model.joblib')
    model_name = 'Calibrated logistic regression'
    model = bundle['model']
    X_test = bundle['X_test']
    y_test = bundle['y_test']

probability = model.predict_proba(X_test)[:, 1]
metrics = pd.Series(
    {
        'ROC-AUC': roc_auc_score(y_test, probability),
        'Brier score': brier_score_loss(y_test, probability),
        'Log loss': log_loss(y_test, probability),
    },
    name=model_name,
)
display(metrics.to_frame('value'))

## Reliability curve

A curve near the diagonal is desirable. Below the diagonal means over-confidence; above the diagonal means under-confidence. Quantile bins give each point a similar number of customers.

In [ ]:
observed_rate, mean_predicted = calibration_curve(
    y_test,
    probability,
    n_bins=10,
    strategy='quantile',
)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(mean_predicted, observed_rate, 'o-', label='Model')
ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
ax.set_title('Reliability curve: ' + model_name)
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Observed churn rate')
ax.legend()
ax.grid(alpha=.25)
plt.tight_layout()
plt.savefig(FIGURES / 'calibration_curve.png', dpi=160)
plt.show()

## What to report

Report Brier score, ROC-AUC, log loss, and whether the high-risk end of the curve is over- or under-confident. If calibration is poor, calibrate on a separate validation process before using probabilities for money decisions.